In [1]:
import json
from pathlib import Path
from collections import defaultdict
import plotly.graph_objects as go

EXCLUDE_CASES = {75, 76, 77, 156, 157, 158, 159}

def load_run(fpath):
    fpath = Path(fpath)
    if not fpath.exists():
        return None
    with open(fpath) as f:
        return [json.loads(l) for l in f]

def get_pred(e):
    compiled = e.get('success') or e.get('lean_verification', {}).get('success', False)
    if not compiled: return 'NC'
    p = str(e.get('prediction', '')).lower()
    if p in ['true', 'yes']: return 'T'
    if p in ['false', 'no']: return 'F'
    return 'Unc'

def get_gt(e):
    g = str(e.get('ground_truth', '')).lower()
    if g in ['true', 'yes']: return 'T'
    if g in ['false', 'no']: return 'F'
    return 'Unc'

def make_gt_pred(gt, pred):
    if pred == 'NC':
        return 'NC'
    return f"{gt}→{pred}"

CATS = ['T→T', 'T→F', 'T→Unc', 'F→T', 'F→F', 'F→Unc', 'Unc→T', 'Unc→F', 'Unc→Unc', 'NC']

NODE_COLOR_MAP = {
    'T→T': 'rgba(39,174,96,1.0)',
    'T→F': 'rgba(39,174,96,0.55)',
    'T→Unc': 'rgba(39,174,96,0.35)',
    'F→T': 'rgba(231,76,60,0.55)',
    'F→F': 'rgba(231,76,60,1.0)',
    'F→Unc': 'rgba(231,76,60,0.35)',
    'Unc→T': 'rgba(243,156,18,0.55)',
    'Unc→F': 'rgba(243,156,18,0.55)',
    'Unc→Unc': 'rgba(243,156,18,1.0)',
    'NC': 'rgba(149,165,166,0.7)',
}

LINK_COLOR_MAP = {
    'T→T': 'rgba(39,174,96,0.4)',
    'T→F': 'rgba(39,174,96,0.25)',
    'T→Unc': 'rgba(39,174,96,0.18)',
    'F→T': 'rgba(231,76,60,0.25)',
    'F→F': 'rgba(231,76,60,0.4)',
    'F→Unc': 'rgba(231,76,60,0.18)',
    'Unc→T': 'rgba(243,156,18,0.25)',
    'Unc→F': 'rgba(243,156,18,0.25)',
    'Unc→Unc': 'rgba(243,156,18,0.4)',
    'NC': 'rgba(149,165,166,0.3)',
}

NODE_COLORS = [NODE_COLOR_MAP[cat] for cat in CATS]

In [2]:
def get_flows(dataset_dir, model_dir, stages, cond_map, base_path='../results'):
    is_folio = 'folio' in dataset_dir
    flows = defaultdict(lambda: defaultdict(int))
    
    for run in [1, 2, 3]:
        data = {}
        for stage, cond in cond_map.items():
            entries = load_run(f'{base_path}/{dataset_dir}/{model_dir}/{cond}_run{run}/results.jsonl')
            if not entries:
                continue
            for e in entries:
                case_idx = e.get('case_idx')
                if is_folio and case_idx in EXCLUDE_CASES:
                    continue
                gt = get_gt(e)
                pred = get_pred(e)
                gt_pred = make_gt_pred(gt, pred)
                if case_idx not in data:
                    data[case_idx] = {}
                data[case_idx][stage] = gt_pred
        
        for case_idx, stages_data in data.items():
            for i in range(len(stages) - 1):
                if stages[i] in stages_data and stages[i+1] in stages_data:
                    src = stages_data[stages[i]]
                    tgt = stages_data[stages[i+1]]
                    flows[(stages[i], stages[i+1])][(src, tgt)] += 1
    return flows

In [3]:
# ADJUST THESE PARAMETERS
NODE_PAD = 5          # Space between nodes (increase for more space)
NODE_THICKNESS = 10   # Node thickness
FONT_SIZE = 11        # Label font size
HEADER_FONT = 11      # Direction header font
STAGE_FONT = 11       # Stage header font
WIDTH = 900           # Plot width
HEIGHT = 400          # Plot height
LEFT_MARGIN = 0      # Left margin
RIGHT_MARGIN = 0     # Right margin
DOMAIN_LEFT = [0.03, 0.47]   # T-direction domain
DOMAIN_RIGHT = [0.53, 0.97]  # F-direction domain

def make_combined_sankey(flows_t, flows_f, stages, title):
    fig = go.Figure()
    
    for side, (flows, domain_x) in enumerate([
        (flows_t, DOMAIN_LEFT),
        (flows_f, DOMAIN_RIGHT)
    ]):
        labels = CATS * len(stages)
        colors = NODE_COLORS * len(stages)
        
        sources, targets, values, link_colors = [], [], [], []
        for i in range(len(stages) - 1):
            for ci, c1 in enumerate(CATS):
                for cj, c2 in enumerate(CATS):
                    v = flows[(stages[i], stages[i+1])].get((c1, c2), 0)
                    if v > 0:
                        sources.append(i * len(CATS) + ci)
                        targets.append((i + 1) * len(CATS) + cj)
                        values.append(v)
                        link_colors.append(LINK_COLOR_MAP[c1])
        
        sankey = go.Sankey(
            node=dict(
                pad=NODE_PAD,
                thickness=NODE_THICKNESS,
                line=dict(color='black', width=0.2),
                label=labels,
                color=colors,
            ),
            link=dict(source=sources, target=targets, value=values, color=link_colors),
            domain=dict(x=domain_x, y=[0.0, 0.82])
        )
        fig.add_trace(sankey)
    
    # Direction headers
    for x_center, dir_label in [((DOMAIN_LEFT[0]+DOMAIN_LEFT[1])/2, 'T-Direction'), 
                                 ((DOMAIN_RIGHT[0]+DOMAIN_RIGHT[1])/2, 'F-Direction')]:
        fig.add_annotation(
            x=x_center, y=0.97,
            text=f"<b>{dir_label}</b>",
            showarrow=False,
            font=dict(size=HEADER_FONT, family='Arial Black'),
            xanchor='center'
        )
    
    # Stage headers
    for side, (x_start, x_end) in enumerate([DOMAIN_LEFT, DOMAIN_RIGHT]):
        for i, stage in enumerate(stages):
            x = x_start + (i / (len(stages) - 1)) * (x_end - x_start)
            fig.add_annotation(
                x=x, y=0.88,
                text=f"<b>{stage}</b>",
                showarrow=False,
                font=dict(size=STAGE_FONT, family='Arial'),
                xanchor='center'
            )
    
    fig.update_layout(
        title=dict(text=title, font=dict(size=11, family='Arial Black'), x=0.5, y=0.95),
        font=dict(size=FONT_SIZE, family='Arial'),
        width=WIDTH,
        height=HEIGHT,
        margin=dict(l=LEFT_MARGIN, r=RIGHT_MARGIN, t=45, b=10),
    )
    
    return fig

In [4]:
STAGES = ['Baseline', 'Directed', 'Nudged']
T_CONDS = {'Baseline': 'baseline', 'Directed': 'bidir_true', 'Nudged': 'spooky_true'}
F_CONDS = {'Baseline': 'baseline', 'Directed': 'bidir_false', 'Nudged': 'spooky_false'}

# Test with one config first
ds, ds_dir, m, m_dir = 'FOLIO', 'folio', 'GPT-5', 'gpt-5'

flows_t = get_flows(ds_dir, m_dir, STAGES, T_CONDS)
flows_f = get_flows(ds_dir, m_dir, STAGES, F_CONDS)
fig = make_combined_sankey(flows_t, flows_f, STAGES, f'{ds} / {m}')
fig.show()

/opt/anaconda3/envs/ada/lib/python3.11/site-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




In [5]:
# Generate all and save
configs = [
    ('FOLIO', 'folio', 'GPT-5', 'gpt-5'),
    ('FOLIO', 'folio', 'DeepSeek', 'deepseek'),
    ('M-LogiEval', 'multilogieval', 'GPT-5', 'gpt-5'),
    ('M-LogiEval', 'multilogieval', 'DeepSeek', 'deepseek'),
]

import os
os.makedirs('../results/prediction_error_fig', exist_ok=True)

for ds, ds_dir, m, m_dir in configs:
    flows_t = get_flows(ds_dir, m_dir, STAGES, T_CONDS)
    flows_f = get_flows(ds_dir, m_dir, STAGES, F_CONDS)
    fig = make_combined_sankey(flows_t, flows_f, STAGES, f'{ds} / {m}')
    fig.write_image(f'../results/prediction_error_fig/sankey_v15_{ds}_{m_dir}.png', scale=2)
    print(f'Saved: sankey_v15_{ds}_{m_dir}.png')

ValueError: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
